# 05 · Grocery category prices (context)

**Milestone 1, item 1.7** · see `docs/MILESTONES.md`

## What we're looking for

Notebook 01 found that ground beef rose about 63% since 2015, while prices in general
(the CPI) rose about 42%. This notebook answers the obvious follow-up: **is beef unusual,
or did all groceries go up this much?** (decision D19)

We use **Consumer Price Index (CPI) categories** from BLS: one index for each part of the
grocery cart. They aren't prices in dollars; each is an index that tracks how the cost of
that category changes over time. To compare them, we rescale every series so its **2015
average = 100**. A value of 140 then means "40% more expensive than in 2015".

**Steps**
1. Setup and API key (same as notebook 01)
2. The categories and their series IDs
3. Download from BLS and save the raw response
4. Tidy table and coverage check
5. Rescale to 2015 = 100 and compare changes (from 2015 and 2019, D20)
6. Chart: meat vs. the rest of the grocery cart
7. Save the table

## Step 1 · Setup and API key

In [1]:
import json
import os
from datetime import date
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import requests
from dotenv import find_dotenv, load_dotenv

env_path = find_dotenv()
load_dotenv(env_path)
BLS_API_KEY = os.getenv("BLS_API_KEY", "").strip()

PROJECT_ROOT = Path(env_path).parent if env_path else Path.cwd().parent
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

print("Project root:", PROJECT_ROOT)
print("BLS key found:", bool(BLS_API_KEY))

Project root: B:\_Dev\Python\meat-prices
BLS key found: True


## Step 2 · The categories

All series are **CPI for All Urban Consumers, U.S. city average, not seasonally
adjusted** (the `CUUR0000` prefix), the same version as the CPI in notebook 01 (D15).
The rest of the ID is the BLS item code.

| Group | Series ID | Category | What's in it |
|---|---|---|---|
| Overall | CUUR0000SA0 | All items | Everything consumers buy (the inflation yardstick) |
| Overall | CUUR0000SAF11 | Food at home | All groceries |
| Overall | CUUR0000SEFV | Food away from home | Restaurants, takeout, cafeterias |
| Grocery category | CUUR0000SAF111 | Cereals and bakery | Bread, flour, rice, pasta, cereal, cookies |
| Grocery category | CUUR0000SAF112 | Meats, poultry, fish and eggs | All protein together |
| Grocery category | CUUR0000SEFJ | Dairy | Milk, cheese, butter, ice cream |
| Grocery category | CUUR0000SAF113 | Fruits and vegetables | Fresh and processed |
| Grocery category | CUUR0000SAF1131 | Fresh fruits and vegetables | Produce |
| Grocery category | CUUR0000SEFM | Processed fruits and vegetables | Canned, frozen and dried |
| Grocery category | CUUR0000SAF114 | Nonalcoholic beverages | Coffee, tea, juice, soda, water |
| Grocery category | CUUR0000SAF115 | Other food at home | Mostly packaged foods: sugar, oils, snacks, frozen and prepared foods, condiments |
| Protein detail | CUUR0000SEFC | Beef and veal | All beef cuts |
| Protein detail | CUUR0000SEFD | Pork | All pork cuts |
| Protein detail | CUUR0000SEFF | Poultry | Chicken and turkey |
| Protein detail | CUUR0000SEFH | Eggs | Eggs |

**Why "Beef and veal" here, when notebook 01 uses ground beef?** The CPI category covers
every beef cut, so it's the fair comparison with other whole categories. Ground beef stays
the headline price in the app; this notebook will show whether the two tell the same
story.

**Food away from home** isn't groceries, but it answers a question readers will ask:
did eating out rise more or less than cooking at home?

In [2]:
SERIES = {
    "CUUR0000SA0":     ("Overall",          "All items"),
    "CUUR0000SAF11":   ("Overall",          "Food at home"),
    "CUUR0000SEFV":    ("Overall",          "Food away from home"),
    "CUUR0000SAF111":  ("Grocery category", "Cereals and bakery"),
    "CUUR0000SAF112":  ("Grocery category", "Meats, poultry, fish and eggs"),
    "CUUR0000SEFJ":    ("Grocery category", "Dairy"),
    "CUUR0000SAF113":  ("Grocery category", "Fruits and vegetables"),
    "CUUR0000SAF1131": ("Grocery category", "Fresh fruits and vegetables"),
    "CUUR0000SEFM":    ("Grocery category", "Processed fruits and vegetables"),
    "CUUR0000SAF114":  ("Grocery category", "Nonalcoholic beverages"),
    "CUUR0000SAF115":  ("Grocery category", "Other food at home (packaged)"),
    "CUUR0000SEFC":    ("Protein detail",   "Beef and veal"),
    "CUUR0000SEFD":    ("Protein detail",   "Pork"),
    "CUUR0000SEFF":    ("Protein detail",   "Poultry"),
    "CUUR0000SEFH":    ("Protein detail",   "Eggs"),
}
START_YEAR, END_YEAR = 2015, date.today().year
print(len(SERIES), "series")

15 series


## Step 3 · Download from BLS

The same request as notebook 01: one call returns all 15 series. If BLS doesn't recognize
a series ID, it says so in a message rather than failing, and the coverage check in Step
4 shows which one.

In [3]:
resp = requests.post("https://api.bls.gov/publicAPI/v2/timeseries/data/", json={
    "seriesid": list(SERIES), "startyear": str(START_YEAR), "endyear": str(END_YEAR),
    "registrationkey": BLS_API_KEY,
}, timeout=60)
resp.raise_for_status()
raw = resp.json()
if raw.get("status") != "REQUEST_SUCCEEDED":
    raise RuntimeError(f"BLS request failed: {raw.get('status')} {raw.get('message')}")

raw_file = RAW_DIR / f"bls_cpi_categories_{date.today():%Y%m%d}.json"
raw_file.write_text(json.dumps(raw, indent=2))

print("Status:", raw["status"])
for msg in raw.get("message", []):
    print("BLS message:", msg)
print("Series returned:", len(raw["Results"]["series"]), "| saved:", raw_file.relative_to(PROJECT_ROOT))

Status: REQUEST_SUCCEEDED
Series returned: 15 | saved: data\raw\bls_cpi_categories_20261026.json


### How to read it
**Status** should be `REQUEST_SUCCEEDED` and **Series returned** 15. A message such as
*"No Data Available for Series X"* means that ID isn't valid; tell me and we'll fix it.

## Step 4 · Tidy table and coverage check

Same parsing as notebook 01: monthly rows only (`M01`–`M12`), values converted to numbers,
oldest first.

**What we're looking for:** every month from January 2015 to the latest month, with one
expected gap: **October 2025**, missing from every BLS series because of the government
shutdown (D18). It's filled during cleaning in Milestone 2, not here.

In [4]:
rows = []
for s in raw["Results"]["series"]:
    for obs in s["data"]:
        if obs["period"].startswith("M") and obs["period"] != "M13":
            rows.append({"series_id": s["seriesID"],
                         "date": pd.Timestamp(int(obs["year"]), int(obs["period"][1:]), 1),
                         "value": pd.to_numeric(obs["value"], errors="coerce")})
cpi = pd.DataFrame(rows)
cpi["group"] = cpi["series_id"].map(lambda s: SERIES[s][0])
cpi["category"] = cpi["series_id"].map(lambda s: SERIES[s][1])
cpi = cpi.sort_values(["series_id", "date"]).reset_index(drop=True)

expected = pd.date_range(f"{START_YEAR}-01-01", cpi["date"].max(), freq="MS")
cov = []
for sid, (group, cat) in SERIES.items():
    have = set(cpi.loc[(cpi["series_id"] == sid) & cpi["value"].notna(), "date"])
    missing = [d.strftime("%Y-%m") for d in expected if d not in have]
    cov.append({"category": cat, "series_id": sid, "months": len(have),
                "missing": ", ".join(missing) if len(missing) <= 3 else f"{len(missing)} months"})
print("Expected months:", len(expected))
pd.DataFrame(cov)

Expected months: 141


,category,series_id,months,missing
0,All items,CUUR0000SA0,140,2025-10
1,Food at home,CUUR0000SAF11,140,2025-10
2,Food away from home,CUUR0000SEFV,140,2025-10
3,Cereals and bakery,CUUR0000SAF111,140,2025-10
4,"Meats, poultry, fish and eggs",CUUR0000SAF112,140,2025-10
5,Dairy,CUUR0000SEFJ,140,2025-10
6,Fruits and vegetables,CUUR0000SAF113,140,2025-10
7,Fresh fruits and vegetables,CUUR0000SAF1131,140,2025-10
8,Processed fruits and vegetables,CUUR0000SEFM,140,2025-10
9,Nonalcoholic beverages,CUUR0000SAF114,140,2025-10


### How to read it
One row per category: how many months have data and which are missing. Every row should
show the expected number of months minus one, with **2025-10** as the only gap.

## Step 5 · Rescale to 2015 = 100 and compare

### First: what the BLS numbers are

A CPI value is an **index level**, not a price or a quantity. It has no units (no dollars,
no pounds). BLS sets each index to **100 in a reference period**, for most series the
average of **1982–1984**, and every later value compares the cost of the same basket of
goods with that period. For example:

- **All items averaged 237.0 in 2015:** the typical basket cost 2.37 times what it cost
  in 1982–84.
- **Beef and veal averaged 322.5 in 2015:** beef cost 3.2 times its 1982–84 level.

Because each index measures change *since its own reference period*, two categories'
levels can't be compared directly. Beef's 322.5 vs. all items' 237.0 doesn't mean beef is
"more expensive"; it means beef rose more *since 1982–84*. Not every series even uses the
same reference period: some categories BLS added later have a different one.

### Rebasing: giving every category the same starting point

We **rebase** each series to our own starting point. For every month:

**rescaled value = (this month's value ÷ that series' 2015 average) × 100**

Worked example with the real values from this download:

| | All items | Beef and veal |
|---|---|---|
| 2015 average (BLS index level) | 237.0 | 322.5 |
| September 2026 (BLS index level) | 336.9 | about 512 |
| Rescaled 2015 | 237.0 ÷ 237.0 × 100 = **100** | 322.5 ÷ 322.5 × 100 = **100** |
| Rescaled September 2026 | 336.9 ÷ 237.0 × 100 = **142** | 512 ÷ 322.5 × 100 = **159** |
| Meaning | 42% more than 2015 | 59% more than 2015 |

After rebasing, every series starts at 100, so the lines can share one chart: anything
above 100 is the percent increase since 2015 (159 → +59%), anything below is a decrease.
The shape of each line doesn't change at all; only the starting point moves.

**Why the 2015 average and not January 2015:** a single month can be unusual; the
average of all twelve months gives a steadier starting point (the same choice as in
notebook 01).

In the code, `avg(2015)` calculates each series' 2015 average, `.map(...)` looks up the
right average for each row by its series ID, and the row's value is divided by it and
multiplied by 100.

Following D20 we show the change from **both 2015 and 2019** (the last year before
COVID), because the starting year changes the story.

In [5]:
avg = lambda year: cpi[cpi["date"].dt.year == year].groupby("series_id")["value"].mean()
cpi["index_2015"] = cpi["value"] / cpi["series_id"].map(avg(2015)) * 100

latest = cpi.dropna(subset=["value"]).sort_values("date").groupby("series_id").tail(1).set_index("series_id")
compare = pd.DataFrame({
    "group": latest["group"], "category": latest["category"],
    "latest_month": latest["date"].dt.strftime("%Y-%m"),
    "change_since_2015_pct": (latest["value"] / avg(2015) - 1) * 100,
    "change_since_2019_pct": (latest["value"] / avg(2019) - 1) * 100,
})
compare.sort_values("change_since_2015_pct", ascending=False).round(1).reset_index(drop=True)

,group,category,latest_month,change_since_2015_pct,change_since_2019_pct
0,Protein detail,Beef and veal,2026-09,58.9,66.8
1,Overall,Food away from home,2026-09,55.7,40.2
2,Grocery category,Nonalcoholic beverages,2026-09,43.0,40.5
3,Overall,All items,2026-09,42.2,31.8
4,Grocery category,Processed fruits and vegetables,2026-09,35.3,37.2
5,Grocery category,Other food at home (packaged),2026-09,35.0,33.9
6,Grocery category,Cereals and bakery,2026-09,34.9,33.6
7,Grocery category,"Meats, poultry, fish and eggs",2026-09,34.7,40.4
8,Protein detail,Poultry,2026-09,33.5,36.8
9,Overall,Food at home,2026-09,32.8,33.1


### How to read it
Every category, sorted from the biggest increase since 2015 to the smallest. **All items**
is the yardstick: anything above it rose faster than prices in general; anything below
rose slower. **Food at home** is the yardstick for groceries specifically.

### What it means
**Beef is at the top of the whole list.** Since 2015, beef and veal rose about **59%**,
more than any grocery category and more than prices in general (**42%**). Since 2019 the
gap is even wider: beef **+67%**, while no other grocery category passed about +40%.

**Groceries as a whole didn't outrun inflation.** Food at home rose about **33%** since
2015, *less* than all items (42%); since 2019 the two are about even (33% vs 32%). So
"food got expensive" mostly means "everything got expensive", except beef.

**Other findings:**
- **Eating out rose much more than cooking at home:** food away from home is up about
  **56%** since 2015 vs. 33% for groceries. (Wages are a large part of restaurant costs.)
- **Packaged foods** ("other food at home", +35%) and **canned and frozen produce** (+35%)
  rose a little more than groceries overall; **fresh produce** (+24%) and **dairy** (+21%)
  rose less.
- **Pork (+26%) and poultry (+34%)** rose less than all items, matching notebook 01.
  (Pork's CPI rose more than the pork-chop price in notebook 01, +9%, because the CPI
  covers every pork product, including bacon and ham.)
- **Eggs show why the baseline matters (D20):** −2% since 2015 but **+37%** since 2019.
  2015 was itself a bird-flu year with very high egg prices, so it's a misleading starting
  point for eggs.

## Step 6 · Chart: meat vs. the rest of the grocery cart

**What we're looking for:** whether beef's line pulls away from the other categories, and
when. The two yardsticks (all items and food at home) are drawn as dashed lines.

In [ ]:
SHOW = {
    "Beef and veal":                  ("#9A3B2E", 2.8, "-"),
    "Pork":                           ("#C97B63", 1.6, "-"),
    "Poultry":                        ("#C08A1E", 1.6, "-"),
    "Eggs":                           ("#E0B84C", 1.2, ":"),
    "Cereals and bakery":             ("#6B8E5A", 1.4, "-"),
    "Dairy":                          ("#7FA7C9", 1.4, "-"),
    "Fresh fruits and vegetables":    ("#4E9A6B", 1.4, "-"),
    "Processed fruits and vegetables":("#2F6690", 1.4, "-"),
    "Other food at home (packaged)":  ("#8C6BB1", 1.4, "-"),
    "Food away from home":            ("#555555", 1.4, "-."),
    "Food at home":                   ("#000000", 1.8, "--"),
    "All items":                      ("#888888", 1.8, "--"),
}

fig, ax = plt.subplots(figsize=(11, 6.5))
for cat, (color, width, style) in SHOW.items():
    d = cpi[cpi["category"] == cat]
    ax.plot(d["date"], d["index_2015"], color=color, linewidth=width, linestyle=style, label=cat)
ax.axhline(100, color="#bbbbbb", linewidth=0.8)
ax.set_ylabel("Index, 2015 average = 100")
ax.set_title("Meat vs. the rest of the grocery cart (2015 = 100)", loc="left", fontweight="bold")
ax.legend(loc="upper left", fontsize=8, ncol=2, frameon=True)
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

### How to read it
Every line starts near 100 in 2015. A line at 150 means that category costs 50% more than
in 2015. Beef is the thick red line; the dashed black and gray lines are food at home and
all items. Eggs are dotted because their spikes (bird flu) would otherwise dominate the
chart. The break in every line in late 2025 is the missing October 2025.

### What it means
Yearly averages of the rebased values tell the story of the chart:

| Year | Beef and veal | Food at home | All items | Food away from home |
|---|---|---|---|---|
| 2019 | 95 | 100 | 108 | 111 |
| 2021 | 114 | 107 | 114 | 120 |
| 2023 | 125 | 125 | 129 | 138 |
| 2025 | **146** | 129 | 136 | 149 |

- Until 2019, beef was actually **cheaper** than in 2015 (95), while prices in general had
  already risen 8%.
- In 2020–2021, beef jumped (COVID plant shutdowns, then rising demand) and caught up
  with all items.
- In 2022–2023, **everything** rose fast (general inflation): groceries, restaurants and
  beef moved together.
- From 2024, **beef broke away on its own**, rising from about 125 to 146 in two years
  while groceries barely moved (125 to 129). That's when the herd hit its low point
  (notebook 03): the beef-specific cause shows up exactly when beef separates from
  everything else.

## Step 7 · Save the table

Saved to `data/processed/` (committed, D9, D12) for the app's "Meat vs. the rest of the
grocery cart" section (item 5.2d).

In [ ]:
out_path = PROCESSED_DIR / "cpi_grocery_categories_monthly.csv"
cpi[["date", "series_id", "group", "category", "value", "index_2015"]].to_csv(out_path, index=False)
print("Saved:", out_path.relative_to(PROJECT_ROOT), f"({len(cpi):,} rows)")

## What's next and why

**Milestone 1 is complete** once this runs: every data source is downloaded and saved.

**Milestone 2: clean and combine** (`06_clean_and_combine.ipynb`). The data now sits in
six processed files with different time periods (monthly, weekly-to-monthly, yearly) and
one known gap (October 2025). Milestone 2 fills that gap (D18), lines everything up on
one monthly calendar, and produces the single table the analysis and the app will use.